# B3 Stage-1 — combined sign DETECTOR (GTSDB + RTSD)

Отдельный ноутбук только для детектора. Проблема: немецкий YOLO **слеп к русским
синим квадратам** (recall 0.03-0.09 на RTSD). Решение — обучить 1-классовый
детектор `sign` на немецких (GTSDB) **и** русских (RTSD) сценах сразу.

GTSDB сильно в меньшинстве (510 кадров против 8485 RTSD ≈ 1:17), поэтому
немецкие кадры **оверсэмплятся** ×`GTSDB_OVERSAMPLE` через список train.txt.

**Чекпойнты пишутся на Google Drive по ходу обучения** (`yolo_runs/`), 
чтобы при обрыве/окончании T4 не потерять прогресс — можно возобновить с
последней эпохи через `resume=True`.

## Один раз ДО запуска
Залей `C:\mitgo\mitgo_det.zip` на Google Drive (`MyDrive/`) именем
`mitgo_det.zip`. Если такое имя уже есть — сначала удали старое.

Runtime → Change runtime type → **T4 GPU**, затем ячейки сверху вниз.

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'GPU НЕ подключён (torch видит CPU). Runtime -> Change runtime type ->\n'
    'T4 GPU -> Save, потом Runtime -> Disconnect and delete runtime,\n'
    'и заново с этой ячейки. Дальше не идти, пока тут не CUDA True.')

In [ ]:
%cd /content
![ -d MITGOTSD ] || git clone https://github.com/damir095/MITGOTSD.git
%cd /content/MITGOTSD
!git pull --ff-only || true
!git log --oneline -1

In [ ]:
!pip -q install ultralytics
import torch; print('torch after pip:', torch.__version__,
                     'CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'pip сбил torch на CPU — Restart runtime и заново'

In [ ]:
# Mount Drive + ПРЕДПРОВЕРКА архива ДО распаковки.
from google.colab import drive
drive.mount('/content/drive')

DET_ZIP   = '/content/drive/MyDrive/mitgo_det.zip'
DET_ROOT  = '/content/det_data'
RUNS_ROOT = '/content/drive/MyDrive/yolo_runs'   # ЧЕКПОЙНТЫ ИДУТ СЮДА

import zipfile, os, pathlib
pathlib.Path(RUNS_ROOT).mkdir(parents=True, exist_ok=True)
assert os.path.exists(DET_ZIP), f'НЕТ файла на Drive: {DET_ZIP}'
_z  = zipfile.ZipFile(DET_ZIP)
_nm = [x.replace(chr(92), '/') for x in _z.namelist()]
_has = lambda p: any(p in x for x in _nm)
print('размер архива :', f'{os.path.getsize(DET_ZIP)/1e6:.0f} MB')
print('entries       :', len(_nm))
print('gtsdb train   :', _has('gtsdb_yolo/images/train/'))
print('rtsd train    :', _has('rtsd_yolo/images/train/'))
print('rtsd val      :', _has('rtsd_yolo/images/val/'))
assert all([_has('gtsdb_yolo/images/train/'),
            _has('rtsd_yolo/images/train/'),
            _has('rtsd_yolo/images/val/')]), 'НЕ ТОТ архив — перезалей mitgo_det.zip'
print('runs root     :', RUNS_ROOT)
print('\nOK — архив правильный.')

In [ ]:
# Распаковка + ОВЕРСЭМПЛ GTSDB через train.txt + генерация yaml.
GTSDB_OVERSAMPLE = 8

import shutil, pathlib, glob
shutil.rmtree(DET_ROOT, ignore_errors=True)
pathlib.Path(DET_ROOT).mkdir(parents=True, exist_ok=True)
_z.extractall(DET_ROOT)

root = pathlib.Path(glob.glob(f'{DET_ROOT}/**/gtsdb_yolo', recursive=True)[0]).parent
g_tr = sorted(glob.glob(f'{root}/gtsdb_yolo/images/train/*'))
r_tr = sorted(glob.glob(f'{root}/rtsd_yolo/images/train/*'))
g_va = sorted(glob.glob(f'{root}/gtsdb_yolo/images/val/*'))
r_va = sorted(glob.glob(f'{root}/rtsd_yolo/images/val/*'))
assert g_tr and r_tr and g_va and r_va, 'пустые сплиты после распаковки'

train_txt = root / 'mitgo_det_train.txt'
val_txt   = root / 'mitgo_det_val.txt'
train_txt.write_text(chr(10).join(r_tr + g_tr * GTSDB_OVERSAMPLE) + chr(10))
val_txt.write_text(chr(10).join(r_va + g_va) + chr(10))

YAML = str(root / 'mitgo_det.yaml')
pathlib.Path(YAML).write_text(
    f'path: {root.as_posix()}\n'
    f'train: {train_txt.as_posix()}\n'
    f'val: {val_txt.as_posix()}\n'
    "nc: 1\nnames: ['sign']\n")
print(pathlib.Path(YAML).read_text())
print(f'RTSD train {len(r_tr)} | GTSDB train {len(g_tr)} x{GTSDB_OVERSAMPLE}'
      f' = {len(g_tr)*GTSDB_OVERSAMPLE}'
      f'  -> train.txt {len(r_tr)+len(g_tr)*GTSDB_OVERSAMPLE} строк')
print(f'val: RTSD {len(r_va)} + GTSDB {len(g_va)} (без оверсэмпла)')

## Обучение
Все веса пишутся в `MyDrive/yolo_runs/combined/weights/` (last.pt каждую
эпоху, best.pt по лучшей val-метрике). При обрыве — см. ячейку **Resume**
ниже, она продолжит с последней сохранённой эпохи.

In [ ]:
# Smoke — 2 эпохи, проверить, что данные/yaml читаются. Веса не нужны.
!cd /content/MITGOTSD && yolo detect train model=yolov8n.pt data={YAML} \
    imgsz=640 batch=16 epochs=2 device=0 project={RUNS_ROOT} name=smoke exist_ok=True

In [ ]:
# Полный прогон. T4 16 GB. ЧЕКПОЙНТЫ ИДУТ НА DRIVE каждую эпоху.
# Если прервётся — НЕ запускай эту ячейку заново (она с нуля), используй
# RESUME-ячейку ниже.
!cd /content/MITGOTSD && yolo detect train model=yolov8n.pt data={YAML} \
    imgsz=640 batch=32 epochs=100 patience=20 device=0 \
    project={RUNS_ROOT} name=combined exist_ok=False

In [ ]:
# RESUME — запускать ТОЛЬКО если предыдущая ячейка прервалась.
# Продолжает с последней сохранённой эпохи (last.pt с Drive).
import os
LAST = f'{RUNS_ROOT}/combined/weights/last.pt'
assert os.path.exists(LAST), f'last.pt не найден: {LAST}'
print('resume from', LAST, f'({os.path.getsize(LAST)/1e6:.1f} MB)')
!cd /content/MITGOTSD && yolo detect train resume=True model={LAST}

In [ ]:
# Финал: best.pt уже на Drive. Просто скачаем себе локально.
BEST = f'{RUNS_ROOT}/combined/weights/best.pt'
import shutil, os
shutil.copy(BEST, '/content/drive/MyDrive/yolo_combined.pt')
print('на Drive: yolo_combined.pt  ', f'({os.path.getsize(BEST)/1e6:.1f} MB)')
from google.colab import files; files.download(BEST)

## Назад на локальную машину
Скачанный **`yolo_combined.pt`** положи в `project/` — дальше я разложу в
`project/runs/detect/experiments/yolo/gtsdb/weights/best.pt` (бэкап старого),
и проверю обе стороны:
* `python -m src.evaluate_b3` — немецкое не просело,
* `python -m src.evaluate_yolo_rtsd` — синие квадраты 43/44/45 теперь видны.